# MapReduce-Reloaded: the demo, end to end

This notebook runs the demo in [`docs/demo.md`](../docs/demo.md) against the
Compose cluster, one claim per section. Sections 7, 8, 10, 11 and 13 check
their claims with `assert`, and 13 checks section 5's output byte for byte;
section 9 checks that its job succeeds, not its counts. The others show
what the cluster did, and any command that fails stops the run:

| § | What it shows | Paper |
|---|---|---|
| 2–5 | Four one-CPU workers; upload, submit and read word count from the host | §2, §3.1 |
| 6 | The answer, plotted | |
| 7 | A job file: map and reduce as Go source, interpreted by the workers | §2 |
| 8 | The combiner: fewer bytes shuffled, the same output | §4.3 |
| 9 | A second program, URL access frequency | §2.3 |
| 10 | A worker killed mid-job: its tasks run again elsewhere | §3.3 |
| 11 | The coordinator restarted mid-job: the job resumes from its log | beyond the paper |
| 12 | The coordinator's metrics | §4.8, §4.9 |
| 13 | Local mode agrees with the cluster byte for byte | §4.7 |
| 14 | The evaluation at a glance: the measured results | §5 |
| 15 | If a cell fails: what to check and run | |

**How the system fits together.** The `mr` client on the host submits a
job to the coordinator, the paper's master, over gRPC. Four workers, each a
container limited to one CPU, pull map and reduce tasks from the
coordinator, send it heartbeats, and fetch each other's map output directly
for the shuffle, so no data passes through the coordinator. Input and output
live in the host's `data/` directory, which every container sees at
`/data`; the coordinator writes every decision to a write-ahead log, so it
can crash and resume. [`design.md`](../design.md#components) has the details.

```text
mr (host) ──submit, status──▶ coordinator ◀──pull tasks, heartbeats── workers 1–4
                              (write-ahead log)                       ⇅ shuffle, worker to worker
          input and output: data/ on the host, /data in every container
```

**Presenting it.** Before each section's code there is a narration note,
set off as a quote: what the cell does, what to point at in its output,
and what to say. *The story in one minute*, below, is the opening.

**Before you start**, on the VM, from the repository root:

```sh
make cluster-up mr      # the cluster and the client
make upload-example     # the corpus and a 10 MB web log
python/.venv/bin/pip install -e 'python[notebook]'   # Jupyter and matplotlib
```

Then open it with `python/.venv/bin/jupyter notebook examples/demo.ipynb`, or
run every cell without a browser:

```sh
python/.venv/bin/jupyter nbconvert --to notebook --execute --inplace examples/demo.ipynb
```

*Kernel → Restart & Run All* takes about two minutes on the course VM, almost
all of it in sections 10 and 11, which sort 1 GiB each (the input is
generated on first use: about 20 seconds and 1 GB of disk). Set
`KILL_WORKER = False` or `RESTART_COORDINATOR = False` in the setup cell to
skip either. Rerunning from the top is safe once the previous run has
finished or stopped: each run uses timestamped output directories, the last
cell removes them, and the setup cell removes any an interrupted run left.
Run one copy at a time against a cluster: the setup cell would also remove
the directories of a copy still running. The status page is at
<http://127.0.0.1:18780/>; keep it open beside the notebook. If a cell
fails, section 15 says what to check.


### How to open this notebook

Everything runs from the prepared checkout, `~/mr-checkout` on the course
VM: launching Jupyter elsewhere leaves §1's `find_root()` pointing at the
wrong tree — and §1 refuses a bare or out-of-date checkout outright, so a
stale clone fails fast with the reason instead of surprising you later. On the VM:

```sh
cd ~/mr-checkout
make cluster-up mr && make upload-example
python/.venv/bin/pip install -e 'python[notebook]'
python/.venv/bin/jupyter notebook --no-browser --port=8888
```

On the laptop, open the same tunnels as [the runbook](../docs/runbook.md)
— Jupyter (8888), the status page (18780) and Grafana (13000) — and follow
the token URL Jupyter prints, which carries its login token:

```sh
ssh -L 8888:127.0.0.1:8888 -L 18780:127.0.0.1:18780 -L 13000:127.0.0.1:13000 mr-vm
```


### The story in one minute

Read this aloud before running anything; every section below is one beat
of it.

1. **The model** (§2–§6). A MapReduce job is two small functions, `Map`
   and `Reduce`. The system does everything else: it splits the input,
   schedules tasks on four one-CPU workers, moves the intermediate data
   between them (the shuffle), and writes sorted output files.
2. **The extensions** (§7–§9). The job can be a Go source file that the
   workers interpret, so nothing is compiled or uploaded. The combiner
   pre-sums each map task's output, so less data crosses the network. A
   second program, URL access frequency, runs on the same machinery.
3. **Fault tolerance** (§10–§11). A worker is killed in the middle of a
   1 GiB sort, and the job still succeeds. Then the coordinator itself is
   crashed in the middle of a sort, which aborts the job in the 2004
   paper. Here the job resumes from the coordinator's write-ahead log and
   writes the same bytes.
4. **Evidence** (§12–§14). The coordinator's own record of every job, a
   byte-for-byte check against an independent single-process run, and the
   measured results of the evaluation.

Every claim is an `assert`, not a screenshot. If the notebook reaches
§16, every claim held on this cluster today.


## 1. Setup


> **Narration: §1, setup.** *Nothing visible happens here; it makes the
> rest of the run trustworthy.*
>
> - **What the cell does.** It defines the helpers the later cells use:
>   `run()` runs a command from the repository root, `submit()` submits a
>   job with `-wait` and records its id, `cat()` reads a job's output, and
>   `status_json()` and `metrics_lines()` read the coordinator's status page
>   and Prometheus metrics. `RUN` is a timestamp, so each run writes to new
>   directories and never collides with an earlier one.
> - **Its checks, in order.** It refuses to start unless this is a git work
>   tree on `main` and current with `origin/main`, `bin/mr` is built, the
>   coordinator answers on port 18780, and at least one worker is live. It
>   also removes anything an interrupted earlier run left behind.
> - **What you should see.** One line: `repository: …, live workers: 4`.
> - **Say.** "Before anything runs, the notebook checks that the code is
>   the code on `main` and that the cluster is up, so nothing below is
>   measured against a stale tree."


In [ ]:
import hashlib
import json
import os
import shutil
import subprocess
import time
import urllib.error
import urllib.request
from pathlib import Path

from IPython.display import Image, Markdown, display

# Sections 10 and 11 each sort 1 GiB: one kills a worker, the other
# restarts the coordinator. False skips a section.
KILL_WORKER = True
RESTART_COORDINATOR = True


def find_root():
    """The repository root: MR_ROOT, or the first of the kernel's working
    directory and its parents that holds deploy/compose. Jupyter starts the
    kernel in examples/, the notebook's directory."""
    if "MR_ROOT" in os.environ:
        return Path(os.environ["MR_ROOT"])
    for path in (Path.cwd(), *Path.cwd().parents):
        if (path / "deploy" / "compose" / "compose.yaml").exists():
            return path
    raise RuntimeError("not inside the repository: set MR_ROOT to its path")


ROOT = find_root()
MR = ROOT / "bin" / "mr"
RUN = f"{int(time.time())}-{os.getpid()}"  # unique per run, even within a second
CORPUS = f"/data/input/notebook-corpus-{RUN}"
COMPOSE = ["docker", "compose", "-p", "mapreduce", "-f", "deploy/compose/compose.yaml"]
METRICS = "http://127.0.0.1:18780/metrics"
STATUS = "http://127.0.0.1:18780/status.json"  # the status page, as data
EXPECTED = b"".join(p.read_bytes() for p in sorted((ROOT / "testdata/corpus/expected").glob("part-r-*")))
JOBS = {}  # this run's job ids and what each ran, for section 12
# A run stopped by a failing cell never reaches the last section's cleanup,
# so clear what earlier runs left before this one adds more.
for old in [*(ROOT / "data/out").glob("notebook-*"), *(ROOT / "data/input").glob("notebook-corpus-*")]:
    shutil.rmtree(old, ignore_errors=True)
SORT_INPUT = ROOT / "data/input/sort-1g.txt"
# The project's chart colors (bench/plot-experiments.py): blue is the
# system as shipped, orange the changed condition, text in ink, not color.
BLUE, ORANGE, INK, INK2 = "#2a78d6", "#eb6834", "#0b0b0b", "#52514e"


def run(*args, check=True, timeout=None):
    """Runs a command in the repository root and returns the result. A
    command that outlasts timeout seconds raises subprocess.TimeoutExpired."""
    result = subprocess.run([str(a) for a in args], cwd=ROOT, text=True, capture_output=True, timeout=timeout)
    if check and result.returncode != 0:
        raise RuntimeError(f"{' '.join(map(str, args))} failed:\n{result.stderr}")
    return result


def submit(label, *args):
    """Submits a job with -wait, records its id under label for section 12,
    and returns its status reply."""
    reply = run(MR, "submit", "-wait", *args).stdout
    job, state = reply.splitlines()[0].split("\t")
    assert state == "SUCCEEDED", reply
    JOBS[job] = label
    return reply


def cat(path):
    """A job's output, its part files in order, as bytes."""
    return subprocess.run([MR, "cat", path], cwd=ROOT, capture_output=True, check=True).stdout


def digest(path):
    """A SHA-256 of a job's part files, in order, read from the host's data/
    directory in chunks (a 1 GiB output does not fit a notebook's memory)."""
    h = hashlib.sha256()
    for part in sorted((ROOT / path.removeprefix("/")).glob("part-r-*")):
        with part.open("rb") as f:
            for chunk in iter(lambda: f.read(1 << 20), b""):
                h.update(chunk)
    return h.hexdigest()


SORT_BYTES = -(-(2**30) // 100) * 100  # bench/gen-sort.py writes whole 100-byte records


def sort_input():
    """Generates the 1 GiB sort input on first use. It is written to a
    temporary file and renamed when complete, and a file of the wrong size,
    such as one left by an interrupted run, is generated again."""
    if SORT_INPUT.exists() and SORT_INPUT.stat().st_size == SORT_BYTES:
        return
    free = shutil.disk_usage(ROOT / "data").free
    assert free > 3 * 2**30, f"{free / 2**30:.1f} GiB free: the sort needs 3 GiB"
    partial = SORT_INPUT.with_name(SORT_INPUT.name + ".partial")
    with partial.open("wb") as out:
        subprocess.run(["python3", "bench/gen-sort.py", str(2**30), "42"], cwd=ROOT, stdout=out, check=True)
    assert partial.stat().st_size == SORT_BYTES, f"{partial} is {partial.stat().st_size} bytes, not {SORT_BYTES}"
    partial.replace(SORT_INPUT)


def live_workers():
    """The number of workers the coordinator counts as live, or 0."""
    try:
        return next((int(line.split()[1]) for line in metrics_lines() if line.startswith("mr_workers ")), 0)
    except (urllib.error.URLError, OSError):
        return 0


def wait_for_workers(n=4, seconds=60):
    """Waits until n workers are live, and returns whether they are."""
    for _ in range(seconds):
        if live_workers() >= n:
            return True
        time.sleep(1)
    return False


def style(axis, title):
    """The project's chart style: a left-aligned finding as the title,
    hairline axes, no box."""
    axis.set_title(title, loc="left", fontsize=11, color=INK)
    for side in ("top", "right"):
        axis.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        axis.spines[side].set_color("#c3c2b7")
    axis.tick_params(colors="#c3c2b7", labelcolor=INK2)


def status_json():
    """The coordinator's status page as data: its jobs and live workers."""
    with urllib.request.urlopen(STATUS, timeout=5) as response:
        return json.load(response)


def containers():
    """Each running container's name, by its short ID, which is also the
    hostname a worker registers with."""
    lines = run("docker", "ps", "--format", "{{.ID}} {{.Names}}").stdout.splitlines()
    return dict(line.split(" ", 1) for line in lines)


def table(header, rows):
    """Shows rows as a Markdown table, which Jupyter renders."""
    lines = ["| " + " | ".join(header) + " |", "|" + "---|" * len(header)]
    lines += ["| " + " | ".join(str(cell) for cell in row) + " |" for row in rows]
    display(Markdown("\n".join(lines)))


def metrics_lines():
    """The coordinator's /metrics, as text lines."""
    with urllib.request.urlopen(METRICS, timeout=5) as response:
        return response.read().decode().splitlines()


def fields(text):
    """A status reply, as a dict: job-1\tSUCCEEDED, map\t4/4, counters."""
    return dict(line.split("\t", 1) for line in text.splitlines() if "\t" in line)


# The demo runs from a work tree on main that is current with origin/main.
# A bare or stale copy (an old clone that lost its work tree) passes
# find_root()'s file test and would run against the wrong tree, so say so.
status = run("git", "-C", ROOT, "rev-parse", "--is-inside-work-tree", check=False).stdout.strip()
assert status == "true", (
    f"{ROOT} is not a git work tree (a bare copy cannot run the demo);"
    " clone the repository and launch this notebook from its examples/ directory")
branch = run("git", "-C", ROOT, "rev-parse", "--abbrev-ref", "HEAD").stdout.strip()
assert branch == "main", f"{ROOT} is on {branch}; the demo runs from main"
try:  # a network that hangs fails here in a minute, not never
    run("git", "-C", ROOT, "fetch", "origin", "main", timeout=60)
except (RuntimeError, subprocess.TimeoutExpired) as error:
    raise RuntimeError(
        f"could not refresh origin/main ({error}); a stale checkout cannot"
        " be verified current") from error
behind = run("git", "-C", ROOT, "rev-list", "--count", "HEAD..origin/main").stdout.strip()
assert behind == "0", (
    f"{ROOT} is {behind} commit(s) behind origin/main;"
    " run `git pull`, then restart the kernel and run again")
assert MR.exists(), "bin/mr is missing: run `make cluster-up mr` first"
try:
    metrics_lines()
except (urllib.error.URLError, OSError) as error:
    raise RuntimeError(f"the coordinator is not answering at {METRICS}: run `make cluster-up`") from error
# A job waits forever for a worker, so stop here if none is live.
workers = live_workers()
if workers == 0:
    raise RuntimeError("the coordinator has no live workers: run `make cluster-up`")
print(f"repository: {ROOT}, live workers: {workers}")


## 2. Check the cluster

One coordinator and four workers, each limited to one CPU. Docker lists the
containers; the coordinator's own view, from its status page, lists the
workers it has registered under its own IDs, which container each is, and
how long ago each last sent a heartbeat. A
worker silent for 10 seconds is declared lost (section 10).


> **Narration: §2, the cluster.** *Show that the hardware matches the
> assignment: one coordinator, four one-CPU workers.*
>
> - **What the cell does.** It asks two sources and shows that they agree.
>   Docker Compose lists five containers. The coordinator's status page
>   (`/status.json`) lists the four workers it has registered, and the cell
>   maps each to its container by hostname.
> - **How to read the table.** A worker's id, such as `r1-w3`, is the
>   coordinator's name for it: `r1` is the coordinator's first life (since
>   the last `make cluster-clean`) and `w3` the third worker to register.
>   The id does not follow the container's number: registration order is a
>   race. *Task slots* is 1 because each worker runs one task at a time on
>   its one CPU. *Last heartbeat* should be under 3 s: workers report every
>   3 s, and three missed reports (10 s) mark a worker lost.
> - **Say.** "Map output stays on the worker that made it. Reducers fetch
>   it from that worker directly, so no data passes through the
>   coordinator: it only schedules."
> - **Next.** Click *Open the status page* below and keep that tab beside
>   the notebook: every job from here on appears there live.


In [ ]:
ps = run(*COMPOSE, "ps", "--format", "{{.Service}}\t{{.Status}}")
print(ps.stdout or "no containers: run `make cluster-up mr`")
print(next((line for line in metrics_lines() if line.startswith("mr_workers ")), "no mr_workers metric"))

status, names = status_json(), containers()
table(
    ["Worker", "Container", "Task slots", "Last heartbeat", "Running"],
    [
        (w["id"], names.get(w["host"], w["host"]), w["slots"], f'{status["at"] - w["last_seen"]:.1f} s ago', len(w["running"]))
        for w in sorted(status["workers"], key=lambda w: w["id"])
    ],
)

In [ ]:
# The workers' live view: the status page refreshes itself every few
# seconds, so the heartbeats, running tasks and progress bars stay current
# through the whole run. On the laptop the same URL rides the SSH forward.
display(Markdown("[Open the status page](http://127.0.0.1:18780/): workers' heartbeats, running tasks and every job's progress bars, refreshed every few seconds."))

## 3. Upload the corpus

Every container sees the host's `data/` directory at `/data`.


> **Narration: §3, upload.** *The client runs on the host, not in a
> container.*
>
> - **What the cell does.** `mr upload` copies the four public-domain books
>   in `testdata/corpus/` (about 0.5 MB) into `data/input/notebook-corpus-…`.
>   The host's `data/` directory is mounted at `/data` in every container,
>   so a path such as `/data/input/...` means the same file to the client,
>   the coordinator and all four workers. This stands in for the paper's
>   GFS (ADR 0016).
> - **What you should see.** The four file names: `alice.txt`,
>   `jekyll.txt`, `raven.txt`, `verwandlung.txt`.


In [ ]:
run(MR, "upload", "testdata/corpus/*.txt", CORPUS)
print(sorted(p.name for p in (ROOT / CORPUS.removeprefix("/")).iterdir()))


## 4. Submit word count

The compiled `wordcount` program, by name: 4 map tasks and 4 reduce tasks.
`-wait` polls the job's status until it ends, and the reply ends with the
job's counters, the paper's §4.9: records and bytes read by the map tasks,
emitted by them, folded by the combiner, and written by the reduce tasks.


> **Narration: §4, word count.** *The programming model of the paper's §2,
> run on the cluster.*
>
> - **What the cell does.** `mr submit -wait` sends the job to the
>   coordinator over gRPC: the compiled `wordcount` program, 4 map tasks,
>   4 reduce tasks. The coordinator cuts the input into 4 splits; each
>   worker asks for a task, maps its split, sorts and combines the result,
>   and keeps one file per reduce task on its own disk. Each reduce task
>   then fetches its partition from every map worker, merges, sums and
>   writes one `part-r-0000N` file.
> - **Read the counters top to bottom.** They tell the story in numbers:
>   9,316 lines read (`map.input.records`), 78,658 words emitted
>   (`map.output.records`), folded to 19,333 by the combiner
>   (`combine.output.records`), reduced to 16,851 distinct words
>   (`reduce.output.records`).
> - **If `backups` is above 0.** It varies from run to run, even on this
>   small job. With as many reduce tasks as workers, the first worker to
>   finish has nothing left to do, so the coordinator gives it a backup copy
>   of a task still running (the paper's §3.6, ADR 0010). `backups_won`
>   counts the backups that finished first. Backups use idle workers, so
>   they cost no time.
> - **If asked about `map_host_local 0`.** Locality (§3.4) counts map tasks
>   that ran on the machine holding their input. Here all input is on one
>   shared directory, so no task is local to one worker.


In [ ]:
WC = f"/data/out/notebook-wc-{RUN}"
reply = submit("word count, compiled", "-m", "4", "-r", "4", "wordcount", WC, f"{CORPUS}/*.txt")
print(reply)
status = fields(reply)
print({key: status[key] for key in ("map", "reduce", "seconds")})


## 5. Read the output


> **Narration: §5, the output.** *The reduce output, read back from the
> host.*
>
> - **What the cell does.** `mr cat` concatenates the four `part-r-*` files
>   in order. Each is sorted by key, as the paper's §4.2 guarantees; the
>   top ten by count is computed here in Python, after the job.
> - **What you should see.** 16,851 distinct words, with "the" first at
>   3,444.


In [ ]:
output = cat(WC)
words = [line.split("\t") for line in output.decode().splitlines()]
top = sorted(((word, int(count)) for word, count in words), key=lambda pair: pair[1], reverse=True)[:10]
print(f"{len(words):,} distinct words")
for word, count in top:
    print(f"{word:>16} {count:>8,}")


## 6. Plot the ten most frequent words


> **Narration: §6, the plot.** *A picture of §5, nothing new.* The title
> states the finding; the bars are the ten counts above. "The" is 4.4% of
> all 78,658 words. Move on quickly.


In [ ]:
import matplotlib.pyplot as plt

words_top, counts_top = [w for w, _ in reversed(top)], [c for _, c in reversed(top)]
figure, axis = plt.subplots(figsize=(7, 3.6))
axis.barh(words_top, counts_top, color=BLUE, height=0.6)
for y, count in enumerate(counts_top):
    axis.annotate(f"{count:,}", (count, y), xytext=(4, 0), textcoords="offset points", va="center", color=INK2, fontsize=9)
axis.set_xlabel("Occurrences in the test corpus", color=INK2)
axis.set_xlim(0, counts_top[-1] * 1.12)
axis.grid(axis="x", color="#e1e0d9", linewidth=1)
axis.set_axisbelow(True)
style(axis, f'"{top[0][0]}" alone is {top[0][1] / sum(int(c) for _, c in words):.1%} of all words')
figure.tight_layout()
plt.show()


## 7. The same job as a job file

`examples/wordcount.go` is word count as Go source. It is sent with the job
and interpreted by the workers (ADR 0015): nothing is compiled or uploaded.
Its output must be the compiled program's, byte for byte.


> **Narration: §7, a job file.** *The extension that matches the
> assignment's "the client submits a file".*
>
> - **What the cell does.** The same word count, but as Go source:
>   [`examples/wordcount.go`](wordcount.go). `mr submit` sends the source
>   text with the job; each worker interprets it with Yaegi, a Go
>   interpreter (ADR 0015). Nothing is compiled and no binary is uploaded,
>   so the workers stay static, unchanging images.
> - **The check.** The output must equal §4's compiled program's byte for
>   byte, and the cell asserts it.
> - **Worth showing.** Open `wordcount.go`: `Map` emits `(word, 1)`,
>   `Reduce` sums, and `var Combine = api.Sum` turns on the combiner.
> - **If asked about speed.** Interpreting costs time: E5 measured 2.3×
>   slower than compiled when the job calls the compiled helpers
>   `api.Words` and `api.Sum`, as this file does, and 10.2× without them.


In [ ]:
JOBFILE = f"/data/out/notebook-jobfile-{RUN}"
jobfile = submit("word count, job file", "-m", "4", "-r", "4", "examples/wordcount.go", JOBFILE, f"{CORPUS}/*.txt")
print({key: value for key, value in fields(jobfile).items() if key in ("map", "reduce", "seconds")})
assert cat(JOBFILE) == output, "the job file and the compiled program differ"
print("job file and compiled program agree: True")


## 8. The combiner: fewer bytes shuffled, the same output

The combiner (§4.3) runs the job's `Combine` on each map task's sorted
output, so a word that occurs a thousand times in a split crosses the
network once, not a thousand times. Here the job file runs again with
`-combiner=false`: the output must not change, and `map.output.bytes`, the
bytes every reducer must fetch, shows what the combiner saves. The saving
grows with how often a word repeats within one map task's input: this
corpus is under 0.5 MB in four splits, so expect a few times fewer bytes
here. At 1 GiB the saving is 15× to 78× ([E1](../bench/README.md#evaluation-e0e7)).


> **Narration: §8, the combiner.** *The paper's §4.3: same answer, less
> network.*
>
> - **What the cell does.** It runs §7's job file again with
>   `-combiner=false` and compares the two jobs' counters.
>   `map.output.bytes` is what the reducers must fetch over the network.
> - **What you should see.** About 198 kB with the combiner against 606 kB
>   without: 3.1× fewer bytes, and the cell asserts that the output is
>   identical. `reduce.input.records` falls the same way (19,333 against
>   78,658).
> - **Say.** "A word that appears a thousand times in one split crosses the
>   network once, not a thousand times. The saving grows with the input:
>   on 1 GiB it was 15× to 78× (E1)."
> - **Timing.** The job seconds here are under a second either way: too
>   small to compare. E1 measured the time saving at 1 GiB (16–42%).


In [ ]:
NOCOMB = f"/data/out/notebook-nocombiner-{RUN}"
on = fields(jobfile)  # section 7's job, combiner on
off = fields(submit("word count, job file, no combiner", "-m", "4", "-r", "4", "-combiner=false", "examples/wordcount.go", NOCOMB, f"{CORPUS}/*.txt"))
assert cat(NOCOMB) == output, "the combiner changed the output"
saved = int(off["map.output.bytes"]) / int(on["map.output.bytes"])
for key in ("map.output.bytes", "reduce.input.records", "seconds"):
    print(f"{key:>22}  on {on[key]:>10}   off {off[key]:>10}")
print(f"the combiner shuffles {saved:.1f}× fewer bytes; the output is identical: True")

figure, axis = plt.subplots(figsize=(7, 2.2))
values = [int(on["map.output.bytes"]) / 1e3, int(off["map.output.bytes"]) / 1e3]
axis.barh(["combiner off", "combiner on"], values[::-1], color=[ORANGE, BLUE], height=0.55)
for y, v in enumerate(values[::-1]):
    axis.annotate(f"{v:,.0f} kB", (v, y), xytext=(4, 0), textcoords="offset points", va="center", color=INK2, fontsize=9)
axis.set_xlabel("Bytes shuffled to the reducers (kB)", color=INK2)
axis.set_xlim(0, max(values) * 1.15)
style(axis, f"The combiner shuffles {saved:.1f}× fewer bytes for the same answer")
figure.tight_layout()
plt.show()


## 9. A second program: the most requested pages


> **Narration: §9, a second program.** *The same machinery runs any `Map`
> and `Reduce`.*
>
> - **What the cell does.** It runs `urlfreq`, one of the paper's §2.3
>   examples, over a 10 MB web log in the Common Log Format. `Map` emits
>   `(url, 1)` for each request; `Reduce` sums. With `-m 2` and no `-r`,
>   it gets 2 map tasks and the default 2 reduce tasks per worker, 8 in all.
> - **What you should see.** `/page1.html` first, with about twice the
>   requests of `/page2.html` and three times those of `/page3.html`: the
>   generator draws pages with probability 1/N, as on a real site.
> - **The check.** Only that the job succeeds; the counts come from a
>   seeded generator, not a golden file.


In [ ]:
assert (ROOT / "data/input/weblog/access.log").exists(), "run `make upload-example` first"
URLS = f"/data/out/notebook-urls-{RUN}"
submit("URL access frequency", "-m", "2", "urlfreq", URLS, "/data/input/weblog/access.log")
urls = [line.split("\t") for line in cat(URLS).decode().splitlines()]
for url, count in sorted(((url, int(n)) for url, n in urls), key=lambda pair: pair[1], reverse=True)[:5]:
    print(f"{count:>8,}  {url}")


## 10. Kill a worker in the middle of a job

The coordinator notices a dead worker after a 10-second heartbeat timeout,
so this uses a job that lasts longer: a 1 GiB sort with 64 map tasks. Five
seconds in, while the job is still running, `docker kill` stops worker 4.
The job still ends `SUCCEEDED`, and the cell requires the coordinator's log
line that this worker, by the ID it registered with, was lost, and how many
of its tasks ran again elsewhere (§3.3 of the paper). The cell then restarts the worker and waits for all
four to be live.


> **Narration: §10, kill a worker.** *The paper's §3.3, worker failure.
> The first of the two moments to slow down for.*
>
> - **What the cell does, step by step.** (1) It submits a 1 GiB sort (64
>   map tasks of 16 MiB) in the background. (2) After 5 s it asserts the
>   job is still running, then kills `mapreduce-worker-4` with
>   `docker kill`: no clean shutdown, as if the machine lost power. (3)
>   The worker stops sending heartbeats; after 10 s the coordinator declares it lost. (4)
>   The lost worker's running task and its *finished* map tasks go back to
>   idle, because their output was on the dead worker's disk. Other workers
>   run them again. (5) The cell restarts the worker and waits for four.
> - **Watch the status page while it runs.** The killed worker's heartbeat
>   age climbs past 10 s; then it moves to the *Lost workers* table, with
>   the time it was lost and the tasks run again, and the map count drops
>   back as its finished map tasks are reset. The restarted container
>   registers as a new worker, under a new id.
> - **What you should see.** `SUCCEEDED`, 64/64 and 4/4, the
>   coordinator's own log line `worker lost … tasks_reset=N` (N was 8 to
>   11 in rehearsals), and all 10,737,419 records written. The job takes
>   about 30–40 s instead of about 13 s: most of the difference is the 10 s
>   detection timeout plus redoing the lost tasks on three workers.
> - **The checks.** The job succeeded; records out equal records in; the
>   log names the killed worker by its coordinator id; at least one task
>   was reset.


In [ ]:
SORT = f"/data/out/notebook-sort-kill-{RUN}"
if not KILL_WORKER:
    print("skipped: KILL_WORKER is False")
else:
    sort_input()
    names = containers()
    killed = next((w["id"] for w in status_json()["workers"] if names.get(w["host"]) == "mapreduce-worker-4"), None)
    assert killed, "mapreduce-worker-4 is not registered with the coordinator: check `docker compose ps`"
    reply = None
    job = subprocess.Popen(
        [MR, "submit", "-m", "64", "-r", "4", "-wait", "distsort", SORT, "/data/input/sort-1g.txt"],
        cwd=ROOT, text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE,
    )
    try:
        time.sleep(5)
        assert job.poll() is None, "the sort finished within 5 s: no worker was lost mid-job"
        since = run("date", "-u", "+%Y-%m-%dT%H:%M:%SZ").stdout.strip()
        run("docker", "kill", "mapreduce-worker-4")
        print(f"killed mapreduce-worker-4, the coordinator's {killed}, five seconds into the job")
        reply, errors = job.communicate()
    finally:
        if reply is None:  # an error above: do not leave the client running
            job.kill()
        run(*COMPOSE, "up", "-d", "--no-build", "worker")
    assert wait_for_workers(), "fewer than 4 live workers after 60 s: check `docker compose ps`"
    assert job.returncode == 0, errors
    status = fields(reply)
    JOBS[reply.split("\t")[0]] = "1 GiB sort, worker 4 killed"
    print(reply.splitlines()[0], {key: status[key] for key in ("map", "reduce", "seconds")})
    assert reply.splitlines()[0].endswith("SUCCEEDED"), reply
    assert status["reduce.output.records"] == status["map.input.records"], "records lost"
    logs = run("docker", "logs", "--since", since, "mapreduce-coordinator-1")
    lost = [line for line in (logs.stdout + logs.stderr).splitlines() if "worker lost" in line and f"worker={killed} " in line]
    assert lost, f"the coordinator did not log losing {killed}, the killed worker"
    reset = int(lost[-1].split("tasks_reset=")[1].split()[0])
    assert reset > 0, lost[-1]
    print(lost[-1])
    print(
        f"The coordinator lost {killed}, the killed worker, and reset {reset} of its tasks; other workers ran them again,"
        f" and the job still wrote all {int(status['reduce.output.records']):,} records in {float(status['seconds']):.1f} s."
    )


## 11. Restart the coordinator in the middle of a job

The paper's master is a single point of failure: if it dies, the job is
aborted (§3.3). Here the coordinator writes every decision to a write-ahead
log before it replies ([ADR 0007](../docs/design/0007-write-ahead-log.md)),
so a restarted coordinator replays the log and finishes the job.

The same 1 GiB sort is submitted without `-wait`. Six seconds in, while the
job is still running, `docker kill` crashes the coordinator and
`docker start` brings it back, as `make e2e` does. (A `docker restart`
would stop it cleanly, and a clean stop tells idle workers to exit.) The job ends
`SUCCEEDED`, and its output is byte for byte the output of section 10's run.
On the course VM a restart costs about 1.2 s
([E4](../bench/README.md#evaluation-e0e7)).


> **Narration: §11, crash the coordinator.** *Beyond the paper. The
> second moment to slow down for.*
>
> - **Why it matters.** In the paper the master is a single point of
>   failure: if it dies, the job is aborted and the client retries. Here
>   the coordinator writes every decision to a write-ahead log on disk
>   before it replies to anyone (ADR 0007).
> - **What the cell does.** It submits the same sort without `-wait` (the
>   client's connection would die with the coordinator), confirms after
>   6 s that the job is `MAPPING` or `REDUCING`, then crashes the
>   coordinator with `docker kill` and brings it back with `docker start`. It polls `mr status` once a second
>   until the job ends; for a few seconds the coordinator is unreachable,
>   which the loop expects.
> - **On the status page.** The page drops for a moment, then comes back
>   with the same job, still in progress, and every earlier job from this
>   run still listed: all replayed from the log. The workers re-register
>   under new ids, `r2-w1` to `r2-w4`, because this is the coordinator's
>   second life.
> - **What you should see.** `SUCCEEDED` within about 20 s, then `same
>   output as the run with a worker killed: True`. That compares SHA-256
>   digests of the two 1 GiB outputs: one job lost a worker, the other
>   lost its coordinator, and both wrote the same bytes.
> - **The measured cost.** E4 put a restart at 1.2 s (15%), with every
>   finished task kept.


In [ ]:
RESTARTED = f"/data/out/notebook-sort-restart-{RUN}"
if not RESTART_COORDINATOR:
    print("skipped: RESTART_COORDINATOR is False")
else:
    sort_input()
    started = time.monotonic()
    job_id = run(MR, "submit", "-m", "64", "-r", "4", "distsort", RESTARTED, "/data/input/sort-1g.txt").stdout.split()[0]
    time.sleep(6)
    before = run(MR, "status", job_id, timeout=10).stdout.splitlines()[0]
    print(before, "<- before the restart")
    assert before.split("\t")[1] in ("PENDING", "MAPPING", "REDUCING"), "the job ended before the restart"
    run("docker", "kill", "mapreduce-coordinator-1")
    run("docker", "start", "mapreduce-coordinator-1")
    print("crashed and started mapreduce-coordinator-1")
    state, reply = "", None
    while time.monotonic() - started < 300:
        try:  # Unavailable while it restarts; a hung request counts as no answer
            reply = run(MR, "status", job_id, check=False, timeout=10)
        except subprocess.TimeoutExpired:
            reply = None
        state = reply.stdout.split("\t", 1)[1].split("\n", 1)[0] if reply and reply.returncode == 0 else ""
        if state in ("SUCCEEDED", "FAILED"):
            break
        time.sleep(1)
    JOBS[job_id] = "1 GiB sort, coordinator restarted"
    print(f"{job_id} {state} after {time.monotonic() - started:.0f} s")
    assert state == "SUCCEEDED", (reply.stdout + reply.stderr) if reply else "no status reply"
    assert wait_for_workers(), "fewer than 4 live workers after 60 s: check `docker compose ps`"
    if KILL_WORKER:
        assert digest(RESTARTED) == digest(SORT), "the restarted job's output differs"
        print("same output as the run with a worker killed: True")


## 12. The coordinator's metrics

Every job this run submitted, as the coordinator reports it on its status
page (state, tasks, time and backup tasks, §3.6), then the raw Prometheus
metrics the status page and the Grafana dashboard are drawn from: live
workers, and completed tasks per job.


> **Narration: §12, the coordinator's view.** *The paper's §4.8 status
> pages and §4.9 counters.*
>
> - **What the cell does.** It reads every job this run submitted from the
>   status page's JSON, beside the label the notebook gave it, then prints
>   the raw Prometheus lines the status page and Grafana are drawn from.
> - **How to read it.** Six jobs, all `succeeded`, with their task counts
>   and times (fewer rows if a section was skipped). *Backups* counts
>   backup attempts (§3.6); they appear even in small jobs, for the reason
>   given in §4. Compare the two sorts: losing a worker costs the 10 s
>   detection timeout plus the redone tasks, while a coordinator restart
>   costs about a second, so the sort with the killed worker is the slower
>   of the two.
> - **Say.** "Every job before §11 survived the coordinator crash: this list
>   is the coordinator's state rebuilt from its log."


In [ ]:
jobs = {job["id"]: job for job in status_json()["jobs"]}
table(
    ["Job", "What it ran", "State", "Map tasks", "Reduce tasks", "Seconds", "Backups"],
    [
        (
            job,
            label,
            jobs[job]["state"],
            f'{jobs[job]["map_completed"]}/{jobs[job]["map_total"]}',
            f'{jobs[job]["reduce_completed"]}/{jobs[job]["reduce_total"]}',
            f'{jobs[job]["duration_seconds"]:.1f}',
            jobs[job]["backup_attempts"],
        )
        for job, label in JOBS.items()
        if job in jobs
    ],
)
for line in metrics_lines():
    if line.startswith("mr_workers ") or ('state="completed"' in line and any(f'job="{job}"' in line for job in JOBS)):
        print(line)

## 13. Local mode is the oracle

Local mode runs the same job in one process, without Docker. Its output, the
cluster's and the expected output checked in at `testdata/corpus/expected`
must all be the same, byte for byte (README, "Run a job on one machine").


> **Narration: §13, the oracle.** *How we know the output is right, not
> just that a job finished.*
>
> - **What the cell does.** `mr run -local` runs the same word count in one
>   process on the host, with no Docker, no network and no coordinator:
>   the paper's §4.7 local execution. Three outputs must then be equal byte
>   for byte: the cluster's (§4), local mode's, and the golden output
>   committed in `testdata/corpus/expected/`.
> - **Say.** "The distributed run, the single-process run and the
>   committed answer all agree. The Python implementation writes the same
>   bytes as well, and `make e2e` and CI check this on every change."


In [ ]:
LOCAL = f"data/out/notebook-local-{RUN}"
run(MR, "run", "-local", "-m", "4", "-r", "4", "wordcount", LOCAL, "testdata/corpus/*.txt")
assert cat(f"/{LOCAL}") == output, "local mode and the cluster differ"
print("local mode and the cluster agree: True")
assert output == EXPECTED, "the cluster's output differs from testdata/corpus/expected"
print("the cluster matches testdata/corpus/expected: True")


## 14. The evaluation at a glance

The paper's §5 experiments and our own, rerun on the course VM with 1 GiB
inputs: medians of five runs, every run's output checked against an
undisturbed run's. `bench/plot-experiments.py` draws them from the data in
`bench/evaluation/`, in order E0 to E7, with the fault recap after E4; [bench/README.md](../bench/README.md#evaluation-e0e7)
has every figure, table and command.


> **Narration: §14, the evaluation.** *What the system does at 1 GiB,
> measured.*
>
> - **What the cell does.** It shows the committed figures from
>   `bench/evaluation/`, each under its one-line finding. Nothing is rerun:
>   the experiments take about two hours (`bench/experiments.sh`).
> - **How they were measured.** On the course VM, 1 GiB inputs, each
>   number the median of five runs with the settings taking turns, and
>   every run's output checked against an undisturbed run's.
> - **The three to linger on.** E0, scaling: word count is 3.7× faster on
>   four workers than on one. E1, the combiner: §8's saving at scale. The
>   faults figure: §10 and §11's costs, measured five times each.
> - **The honest caveat.** Four co-located one-CPU workers sharing one
>   disk are not 1,800 machines. Fixed costs such as startup, heartbeats
>   and the 10 s failure timeout are a large share of an 8-second job,
>   which is why a lost worker costs proportionally more here than in the
>   paper ([evaluation](../docs/evaluation.md)).
> - `THEME = "-dark"` shows the dark variants if the projector needs them.


In [ ]:
THEME = ""  # "-dark" for the dark variants
# Each figure's finding, as README.md's results table states it.
FIGURES = {
    "E0": "E0, scaling over workers: word count runs 3.7× faster on 4 workers than on 1, distributed sort 3.1×, grep 2.0×.",
    "E1": "E1, the combiner: 15× to 78× fewer bytes shuffled and 16–42% less job time, more as the keys get more skewed.",
    "E2": "E2, backup tasks: with one worker at 0.2 CPU, backups cut 17.9 s to 11.4 s; with no straggler they cost nothing measurable.",
    "E3": "E3, losing a worker: every job finishes with the undisturbed run's output, +8 s to +26 s in all.",
    "E4": "E4, a coordinator restart: recovery from the log costs 1.2 s (15%) and keeps every finished task.",
    "faults": "E2 to E4 together: what each fault costs.",
    "E5": "E5, an interpreted job file: 2.3× slower than compiled with the helpers api.Words/api.Sum, 10.2× without them.",
    "E6": "E6, the combiner on real text: 5.2× fewer bytes and 11% less job time on crawled English.",
    "E7": "E7, the paper's backups against LATE: either rule cuts a sort on mixed-speed workers by 32%; LATE needs fewer backups, and the two times overlap.",
}
for name, finding in FIGURES.items():
    display(Markdown(f"**{finding}**"))
    display(Image(filename=str(ROOT / "bench/evaluation" / f"{name}{THEME}.png"), width=760))

### The takeaway

If the run reached here, every claim above held on this cluster today:

- the programming model works, on four workers, from a compiled program or
  an interpreted Go file;
- the combiner cut the bytes shuffled without changing the answer;
- a worker killed mid-job and a coordinator crashed mid-job both left the
  job `SUCCEEDED` with identical output, the second beyond what the 2004
  paper could do;
- the cluster, local mode and the committed golden output agree byte for
  byte.

§15 is for when something goes wrong; §16 removes this run's files.


## 15. If a cell fails

Every check stops with a message that names its cause. Rerunning from the
top is safe once no other copy is running: setup clears what an earlier or
interrupted run left, including directories a running copy still uses.

| Message or symptom | What to run |
|---|---|
| `bin/mr is missing`, `not answering`, `no live workers` | `make cluster-up mr`, then `docker compose -p mapreduce -f deploy/compose/compose.yaml ps` |
| `fewer than 4 live workers after 60 s` | `docker compose -p mapreduce -f deploy/compose/compose.yaml up -d --no-build worker` |
| ``run `make upload-example` first`` | `make upload-example` |
| `GiB free: the sort needs 3 GiB` | free disk space, or set `KILL_WORKER` and `RESTART_COORDINATOR` to `False` |
| `the sort finished within 5 s`, `the job ended before the restart` | the machine is faster than the course VM: rerun the cell, or skip it with its flag |
| Anything else | `make demo-check`: one `OK`, `WARN` or `FAIL` line per thing the demo needs, each `FAIL` with what to run |


## 16. Clean up

This run's input and output. The sort input stays in `data/input/` for the next run.


In [ ]:
# A section that did not run leaves its name undefined, so clean up only
# the paths the notebook actually created (the setup always defines CORPUS).
for name in ("WC", "JOBFILE", "NOCOMB", "URLS", "SORT", "RESTARTED", "LOCAL", "CORPUS"):
    path = globals().get(name)
    if path:
        shutil.rmtree(ROOT / path.removeprefix("/"), ignore_errors=True)
print("removed this run's input and output")
